# Model Comparison & Hyperparameter Tuning
**Objective:** Compare tree-based ensemble algorithms against the linear baseline, tune hyperparameters using randomized search to prevent overfitting, and serialize the final unbiased model for production.

## 1. Setup & Pre-split Data Loading
To guarantee strict reproducibility and prevent data leakage, we load the statically split training and testing sets exported from the previous phase, alongside our robust `ColumnTransformer`.

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, make_scorer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

import warnings
warnings.filterwarnings('ignore')

# 1. LOAD THE PRE-SPLIT DATA (Your Idea!)
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
# .squeeze() converts it from a 1-column DataFrame back into a Pandas Series
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze() 

# 2. DEFINE THE PREPROCESSOR (Same as Phase 5)
nominal_cols = ['Brand', 'FuelType', 'Transmission']
ordinal_cols = ['Owner']
num_cols = ['Age', 'kmDriven']
owner_categories = [['first', 'second']]

preprocessor = ColumnTransformer(
    transformers=[
        ('nom', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_cols),
        ('ord', OrdinalEncoder(categories=owner_categories, handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cols),
        ('num', StandardScaler(), num_cols)
    ],
    remainder='drop'
)

## 2. Algorithm Comparison via Cross-Validation
Evaluating Bagging (`RandomForest`) vs. Boosting (`GradientBoosting`). 
*Note: Because our target is log-transformed (`AskPrice_Log`), we implement a custom scoring function (`inverse_log_mae`) to calculate Cross-Validation error in actual Rupees.*

In [5]:
# RUN CROSS-VALIDATION (Bagging vs Boosting vs Linear)
def inverse_log_mae(y_true_log, y_pred_log):
    y_true_rupees = np.expm1(y_true_log)
    y_pred_rupees = np.expm1(y_pred_log)
    return mean_absolute_error(y_true_rupees, y_pred_rupees)

rupee_mae_scorer = make_scorer(inverse_log_mae, greater_is_better=False)

models = {
    "Baseline (Linear)": LinearRegression(),
    "Bagging (Random Forest)": RandomForestRegressor(random_state=42),
    "Boosting (Gradient Boosting)": GradientBoostingRegressor(random_state=42)
}

print("Running 5-Fold Cross-Validation. This may take a minute or two...\n")

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    scores = cross_val_score(
        pipeline, X_train, y_train, cv=5, 
        scoring=rupee_mae_scorer, n_jobs=-1
    )
    
    mean_mae = -scores.mean()
    print(f"{name}:")
    print(f"  -> Mean MAE: ₹{mean_mae:,.0f}")

Running 5-Fold Cross-Validation. This may take a minute or two...

Baseline (Linear):
  -> Mean MAE: ₹339,608
Bagging (Random Forest):
  -> Mean MAE: ₹314,538
Boosting (Gradient Boosting):
  -> Mean MAE: ₹331,075


## 3. Hyperparameter Tuning (Randomized Search)
`RandomForest` outperformed the baseline out-of-the-box. We now use `RandomizedSearchCV` to optimize its hyperparameters (e.g., limiting `max_depth` to prevent infinite depth memorization) while minimizing computational cost.


In [6]:
from sklearn.model_selection import RandomizedSearchCV

# 1. Define the Hyperparameter Space
# The "regressor__" prefix tells the Pipeline to pass these to the RandomForest
param_distributions = {
    'regressor__n_estimators': [100, 200, 300],     # Number of trees
    'regressor__max_depth': [10, 15, 20, 25],       # How deep they can grow
    'regressor__min_samples_split': [2, 5, 10],     # Min cars needed to split a node
    'regressor__min_samples_leaf': [1, 2, 4]        # Min cars required at the very end of a branch
}

# 2. Re-instantiate our best Pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

# 3. Setup the Randomized Search
print("Starting Randomized Search (testing 10 combinations)...")
random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_distributions,
    n_iter=10,                      # Try 10 random combinations
    cv=3,                           # 3-Fold CV for speed
    scoring=rupee_mae_scorer,       # Use our custom Rupee error!
    random_state=42,        
    n_jobs=-1,                      # Use all CPU cores
    verbose=2                       # Print progress
)

# 4. Execute the Search
random_search.fit(X_train, y_train)

# 5. View Results
best_rf_pipeline = random_search.best_estimator_

print("\n--- Tuning Complete ---")
print(f"Best Hyperparameters: {random_search.best_params_}")
# Note: we multiply by -1 because the scorer returns negative numbers
print(f"New Best CV MAE: ₹{-random_search.best_score_:,.0f}")

Starting Randomized Search (testing 10 combinations)...
Fitting 3 folds for each of 10 candidates, totalling 30 fits

--- Tuning Complete ---
Best Hyperparameters: {'regressor__n_estimators': 300, 'regressor__min_samples_split': 10, 'regressor__min_samples_leaf': 1, 'regressor__max_depth': 20}
New Best CV MAE: ₹310,528


## 4. Unbiased Final Evaluation & Serialization
To avoid Optimization Bias, the test set (`X_test`) has been strictly held out until this moment. We evaluate the tuned pipeline on this unseen data to establish our final V1 metrics, and serialize the pipeline using `joblib` for downstream error analysis and deployment.

In [7]:
import joblib
import os
from sklearn.metrics import mean_squared_error


# 1. Predict on the UNTOUCHED Test Set
y_test_pred_log = best_rf_pipeline.predict(X_test)

# 2. Convert back to actual Rupees
y_test_actual_rupees = np.expm1(y_test)
y_test_pred_rupees = np.expm1(y_test_pred_log)

# 3. Calculate Final Unbiased Metrics
final_mae = mean_absolute_error(y_test_actual_rupees, y_test_pred_rupees)
final_rmse = np.sqrt(mean_squared_error(y_test_actual_rupees, y_test_pred_rupees))

print("--- FINAL TEST SET PERFORMANCE ---")
print(f"Test MAE:  ₹{final_mae:,.0f}")
print(f"Test RMSE: ₹{final_rmse:,.0f}\n")

# 4. Serialize (Save) the Pipeline
os.makedirs('../models', exist_ok=True)
joblib.dump(best_rf_pipeline, '../models/rf_tuned_pipeline.joblib')
print("Model successfully saved to ../models/rf_tuned_pipeline.joblib")

--- FINAL TEST SET PERFORMANCE ---
Test MAE:  ₹309,522
Test RMSE: ₹1,169,145

Model successfully saved to ../models/rf_tuned_pipeline.joblib
